# Portfolio peak and evening-share report — 2023 settlement data

Build hourly UTC load per meter from the half-hourly settlement file, produce the
portfolio peak, the evening share and a per-meter annual reconciliation against the
contracted estimates.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

hh = pd.read_csv("../data/meter_halfhourly_2023.csv.gz")
meters = pd.read_csv("../data/meters.csv")
hh.shape

(349439, 4)

Settlement data is stamped with a date and a period number (1–48). Convert to UTC timestamps.

In [2]:
hh["utc"] = pd.to_datetime(hh["settlement_date"]) + pd.to_timedelta((hh["settlement_period"] - 1) * 30, unit="min")
hh = hh.sort_values(["meter_id", "utc"]).reset_index(drop=True)
hh.head()

,meter_id,settlement_date,settlement_period,kwh,utc
0,M100000,2023-01-01,1,0.734,2023-01-01 00:00:00
1,M100000,2023-01-01,2,0.638,2023-01-01 00:30:00
2,M100000,2023-01-01,3,0.394,2023-01-01 01:00:00
3,M100000,2023-01-01,4,0.369,2023-01-01 01:30:00
4,M100000,2023-01-01,5,0.528,2023-01-01 02:00:00


In [3]:
hh["settlement_period"].describe()

count    349439.000000
mean         24.500030
std          13.853989
min           1.000000
25%          12.000000
50%          25.000000
75%          36.000000
max          50.000000
Name: settlement_period, dtype: float64

## Hourly load per meter

In [4]:
hourly = (hh.set_index("utc")
            .groupby("meter_id")["kwh"]
            .resample("h").mean()
            .rename("kwh_per_hour")
            .reset_index())
hourly.head()

,meter_id,utc,kwh_per_hour
0,M100000,2023-01-01 00:00:00,0.6860
1,M100000,2023-01-01 01:00:00,0.3815
2,M100000,2023-01-01 02:00:00,0.8095
3,M100000,2023-01-01 03:00:00,0.7435
4,M100000,2023-01-01 04:00:00,0.7990


In [5]:
hourly.groupby("meter_id")["kwh_per_hour"].agg(["count", "mean", "max"]).round(3).head(8)

,count,mean,max
meter_id,,,
M100000,8759,1.289,6.580
M100001,8759,0.136,0.692
M100002,8759,0.217,1.220
M100003,8759,0.155,0.726
M100004,8759,0.131,0.542
M100005,8759,0.180,0.762
M100006,8759,0.244,1.031
M100007,8759,16.282,602.500


## Largest customers

In [6]:
annual = hh.groupby("meter_id")["kwh"].sum().rename("annual_kwh").sort_values(ascending=False)
annual.head(5).round(0)

meter_id
M100007    285046.0
M100015     38296.0
M100010     22929.0
M100000     22566.0
M100016      5502.0
Name: annual_kwh, dtype: float64

## Peak demand

Peak kW per meter is the maximum hourly load. The portfolio capacity requirement is the
sum of the individual peaks.

In [7]:
peak_kw = hourly.groupby("meter_id")["kwh_per_hour"].max().rename("peak_kw")
portfolio_peak_kw = peak_kw.sum()
print(f"portfolio peak requirement: {portfolio_peak_kw:,.1f} kW")
peak_kw.sort_values(ascending=False).head(5).round(2)

portfolio peak requirement: 641.3 kW


meter_id
M100007    602.50
M100015     10.63
M100000      6.58
M100010      6.52
M100016      1.48
Name: peak_kw, dtype: float64

## Evening share

Share of energy consumed in the 17:00–20:00 evening window. Benchmark from the retail team
is 22% for the residential book.

In [8]:
hourly["hour"] = hourly["utc"].dt.hour
evening = hourly[hourly["hour"].between(17, 20)]
evening_share = evening["kwh_per_hour"].sum() / hourly["kwh_per_hour"].sum()
print(f"evening share: {evening_share:.1%}  (benchmark 22%)")

evening share: 24.9%  (benchmark 22%)


In [9]:
by_month = hourly.assign(month=hourly["utc"].dt.month)
share_m = (by_month[by_month["hour"].between(17, 20)].groupby("month")["kwh_per_hour"].sum()
           / by_month.groupby("month")["kwh_per_hour"].sum())
share_m.round(3)

month
1     0.190
2     0.191
3     0.193
4     0.192
5     0.195
6     0.196
7     0.197
8     0.195
9     0.276
10    0.193
11    0.192
12    0.191
Name: kwh_per_hour, dtype: float64

## Annual reconciliation against contracted estimates

In [10]:
m = meters.copy()
m["meter_id"] = m["meter_id"].where(m.index % 7 != 3, m["meter_id"].str.lower())   # ids as exported by the CRM

recon = (annual.reset_index().assign(meter_id=lambda d: d["meter_id"].str.upper())
           .merge(m[["meter_id", "customer_type", "annual_kwh_estimate"]], on="meter_id", how="inner"))
recon["ratio"] = recon["annual_kwh"] / recon["annual_kwh_estimate"]
recon.sort_values("ratio").round(2)

,meter_id,annual_kwh,customer_type,annual_kwh_estimate,ratio
16,M100013,1509.93,residential,2948.0,0.51
4,M100011,4544.88,residential,4511.0,1.01
8,M100002,3805.46,residential,3665.0,1.04
3,M100016,5502.35,residential,5283.0,1.04
12,M100005,3151.98,residential,3026.0,1.04
1,M100015,38295.80,sme,36741.0,1.04
5,M100006,4265.08,residential,4089.0,1.04
14,M100001,2385.65,residential,2286.0,1.04
2,M100000,22565.63,sme,21622.0,1.04
10,M100009,3697.64,residential,3541.0,1.04


In [11]:
print("meters reconciled:", len(recon))
print("median ratio actual/estimate:", round(recon["ratio"].median(), 3))
under = recon[recon["ratio"] < 1.02]
under[["meter_id", "annual_kwh", "annual_kwh_estimate", "ratio"]].round(3)

meters reconciled: 17
median ratio actual/estimate: 1.044


,meter_id,annual_kwh,annual_kwh_estimate,ratio
4,M100011,4544.881,4511.0,1.008
16,M100013,1509.926,2948.0,0.512


## Results

- Portfolio peak requirement: see above (sum of meter peaks).
- Evening share below the 22% benchmark: the book is less evening-heavy than assumed.
- M100007 is by far the largest customer and dominates the portfolio peak.
- All meters reconcile to within a few % of their contracted estimate; M100011 is the only one at or below
  its estimate (~4% under its peers): an efficiency gain worth flagging to the account team.

In [12]:
pd.Series({
    "portfolio_peak_kw": round(portfolio_peak_kw, 1),
    "evening_share": round(evening_share, 4),
    "largest_meter": annual.index[0],
    "largest_meter_kwh": round(annual.iloc[0]),
    "meters_reconciled": len(recon),
    "median_actual_vs_estimate": round(recon["ratio"].median(), 3),
})

portfolio_peak_kw              641.3
evening_share                 0.2494
largest_meter                M100007
largest_meter_kwh             285046
meters_reconciled                 17
median_actual_vs_estimate      1.044
dtype: object